# Higgs Boson Signal Classification with LightGBM
DATA 400 Individual Mini-Project (physics focus)

**Goal:** separate signal (`s`, Higgs decaying to tau tau) from background (`b`) in the ATLAS Higgs ML Challenge 2014 data, and evaluate with AUC and AMS.

**Pipeline:** load data, inspect weights, train LightGBM, evaluate (ROC and AMS), write a Kaggle-style submission.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from src.lgbm_model import train_lgbm, plot_roc, make_submission

## 2. Load data
Swap the `read_csv` line for your `DataLoader` call if you prefer. The full CERN file has a `KaggleSet` column (`t` = training, `b`/`v` = public/private test, `u` = unused); check the values in your copy.

In [ ]:
df = pd.read_csv("data/atlas-higgs-challenge-2014-v2.csv")   # adjust path or use DataLoader

train_df = df[df["KaggleSet"] == "t"].reset_index(drop=True)
test_df  = df[df["KaggleSet"].isin(["b", "v"])].reset_index(drop=True)

print(train_df.shape, test_df.shape)
train_df.head()

## 3. Inspect labels and event weights
Weights are per event. Background weights span a much wider range than signal weights, which is why weighting matters for both training and evaluation.

In [ ]:
print(train_df["Label"].value_counts())
print()
print(train_df[["EventId", "Label", "Weight"]].head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for lab, color in [("s", "tab:red"), ("b", "tab:blue")]:
    w = train_df.loc[train_df["Label"] == lab, "Weight"]
    ax.hist(np.log10(w), bins=50, alpha=0.6, label=f"{lab} (n={len(w)})", color=color)
ax.set_xlabel("log10(Weight)")
ax.set_ylabel("Events")
ax.set_title("Event weight distribution by class")
ax.legend()
plt.show()

## 4. Train LightGBM
`train_lgbm` makes a stratified 80/20 split, trains with early stopping on weighted validation AUC, then scans thresholds for the best AMS on the validation set.

In [ ]:
model, thr, val = train_lgbm(train_df)

## 5. Evaluate

### 5a. ROC curve (weighted vs. unweighted)

In [ ]:
ax = plot_roc(val, weighted=True)
plot_roc(val, weighted=False, ax=ax)
plt.show()

### 5b. Validation predictions alongside their weights

In [ ]:
pd.DataFrame(val).head(10)   # y = true label (1 = signal), p = predicted score, w = event weight

### 5c. Feature importance

In [ ]:
lgb.plot_importance(model, importance_type="gain", max_num_features=15, figsize=(7, 6))
plt.title("Top 15 features by gain")
plt.tight_layout()
plt.show()

## 6. Kaggle-style submission
The test rows have no usable labels for scoring here, so this file only contains `EventId`, `RankOrder`, and `Class`.

In [ ]:
submission = make_submission(model, test_df, thr, path="submission.csv")
submission.head()

## 7. Notes and next steps
- Compare against the XGBoost baseline on the same split (AMS, AUC, training time).
- Record the best AMS threshold and best iteration reported above in the write-up.
- Optional: early stop on AMS with a custom `feval` instead of AUC and compare.